In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
property_gdf = gpd.read_parquet("../../data/processed/nsw_property/property.parquet")
flood_gdf = gpd.read_parquet("../../data/processed/nsw_flood/flood.parquet")
site_features = gpd.read_parquet("../../data/processed/site_features/property_site_features_zoning_heritage_bushfire_sample.parquet")

In [3]:
print("Property rows:", len(property_gdf))
print("Flood rows:", len(flood_gdf))
print("Site rows:", len(site_features))

print("Property CRS:", property_gdf.crs)
print("Flood CRS:", flood_gdf.crs)
print("Site CRS:", site_features.crs)

Property rows: 4198396
Flood rows: 622
Site rows: 49950
Property CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation":

In [4]:
property_main = property_gdf[property_gdf["propertytype"] == 1].copy()
print(len(property_main))

4193334


In [5]:
flood_keep = flood_gdf[
    [
        "OBJECTID",
        "EPI_NAME",
        "LGA_NAME",
        "LAY_CLASS",
        "EPI_TYPE",
        "COMMENT",
        "geometry",
    ]
].copy()

In [6]:
if property_main.crs != flood_keep.crs:
    property_main = property_main.to_crs(flood_keep.crs)

In [7]:
sample_rids = set(site_features["RID"].unique())
property_sample = property_main[property_main["RID"].isin(sample_rids)].copy()
print(len(property_sample))

49950


In [8]:
joined = gpd.sjoin(
    property_sample,
    flood_keep,
    how="left",
    predicate="intersects"
)

In [9]:
print("Joined rows:", len(joined))
joined[
    ["RID", "address", "LAY_CLASS", "COMMENT", "EPI_NAME", "LGA_NAME"]
].head(10)

Joined rows: 50162


,RID,address,LAY_CLASS,COMMENT,EPI_NAME,LGA_NAME
173,260,44 NORTHCOTE STREET ABERDARE,NaN,NaN,NaN,NaN
217,319,10 MEREWETHER CLOSE NORTH ROTHBURY,NaN,NaN,NaN,NaN
221,327,112 MATHIESON STREET BELLBIRD HEIGHTS,NaN,NaN,NaN,NaN
232,338,584 WOLLOMBI ROAD BELLBIRD,NaN,NaN,NaN,NaN
251,359,41 RUSSELL STREET BRANXTON,NaN,NaN,NaN,NaN
285,400,14 FOSTER STREET CESSNOCK,NaN,NaN,NaN,NaN
327,456,13 WILLIAM STREET CESSNOCK,NaN,NaN,NaN,NaN
415,606,761 WATAGAN CREEK ROAD WATAGAN,NaN,NaN,NaN,NaN
616,913,4 CORAL CRESCENT PEARL BEACH,NaN,NaN,NaN,NaN
652,951,65 DONALD AVENUE UMINA BEACH,NaN,NaN,NaN,NaN


In [10]:
joined["LAY_CLASS"].isna().mean()

np.float64(0.987002113153383)

In [11]:
joined["LAY_CLASS"].value_counts(dropna=False).head(20)

LAY_CLASS
NaN                                  49510
Flood Planning Area                    316
Probable Maximum Flood Line            228
Flood Prone and Major Creeks Land      105
1 in 100 AEP Flood Extent                2
Transitional Land                        1
Name: count, dtype: int64

In [12]:
joined["RID"].duplicated().mean()

np.float64(0.004226306766077908)

In [13]:
def collapse_flood_group(group: pd.DataFrame) -> pd.Series:
    lay_nonnull = group["LAY_CLASS"].dropna()
    comment_nonnull = group["COMMENT"].dropna()

    flood_classes = sorted(lay_nonnull.astype(str).unique().tolist()) if len(lay_nonnull) else []
    flood_comments = sorted(comment_nonnull.astype(str).unique().tolist()) if len(comment_nonnull) else []

    if len(lay_nonnull):
        primary_class = lay_nonnull.value_counts().index[0]
    else:
        primary_class = pd.NA

    return pd.Series(
        {
            "flood_flag": int(group["LAY_CLASS"].notna().any()),
            "flood_hit_count": int(group["LAY_CLASS"].notna().sum()),
            "flood_class_count": int(lay_nonnull.nunique()),
            "flood_classes": "|".join(flood_classes),
            "flood_comments": "|".join(flood_comments),
            "primary_flood_class": primary_class,
        }
    )

In [14]:
flood_features = (
    joined.groupby("RID", as_index=False)
    .apply(collapse_flood_group)
    .reset_index(drop=True)
)

print(len(flood_features))
flood_features.head()

49950


/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_41953/1273663410.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_flood_group)


,RID,flood_flag,flood_hit_count,flood_class_count,flood_classes,flood_comments,primary_flood_class
0,260,0,0,0,,,<NA>
1,319,0,0,0,,,<NA>
2,327,0,0,0,,,<NA>
3,338,0,0,0,,,<NA>
4,359,0,0,0,,,<NA>


In [15]:
site_with_flood = site_features.merge(
    flood_features,
    on="RID",
    how="left",
)

print(len(site_with_flood))
site_with_flood.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,primary_bushfire_category,bushfire_risk_level,has_bushfire_cat1,has_bushfire_buffer,flood_flag,flood_hit_count,flood_class_count,flood_classes,flood_comments,primary_flood_class
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,None,0,0,0,0,0,0,,,<NA>
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,Vegetation Category 1,3,1,1,0,0,0,,,<NA>
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,None,0,0,0,0,0,0,,,<NA>
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,Vegetation Buffer,1,0,1,0,0,0,,,<NA>
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,None,0,0,0,0,0,0,,,<NA>


In [16]:
site_with_flood["flood_flag"] = site_with_flood["flood_flag"].fillna(0).astype(int)
site_with_flood["flood_hit_count"] = site_with_flood["flood_hit_count"].fillna(0).astype(int)
site_with_flood["flood_class_count"] = site_with_flood["flood_class_count"].fillna(0).astype(int)

In [17]:
print(site_with_flood["flood_flag"].value_counts(normalize=True, dropna=False))
print(site_with_flood["primary_flood_class"].value_counts(dropna=False).head(20))
print(site_with_flood["flood_class_count"].describe())

flood_flag
0    0.991191
1    0.008809
Name: proportion, dtype: float64
primary_flood_class
<NA>                                 49510
Flood Planning Area                    305
Flood Prone and Major Creeks Land       96
Probable Maximum Flood Line             36
1 in 100 AEP Flood Extent                2
Transitional Land                        1
Name: count, dtype: int64
count    49950.000000
mean         0.012653
std          0.142059
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          2.000000
Name: flood_class_count, dtype: float64


In [19]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_with_flood.to_parquet(
    "../../data/processed/site_features/property_site_features_zoning_heritage_bushfire_flood_sample.parquet",
    index=False,
)